In [25]:
"""
Outbreak check — parse BEAST2 tree files and summarise tree/subtree widths.

Cell 0: Parse dataset and print original summary.
Cell 1: Filter out outbreak trees (subtree_width in top X%) and re-summarise.
"""

import re
import numpy as np
from pathlib import Path
from collections import defaultdict

# --- Configuration -------------------------------------------------------------
DATASET = '/Users/lukelyu/Desktop/epidata/3k/'

# Regex to extract individual trees and tip locations from BEAST2 .trees files
TREE_RE = re.compile(r'tree STATE_\d+ = (.+?)(?=\ntree |\nEnd;|$)', re.DOTALL)
TIP_RE  = re.compile(r'\d+\[&type="I\{(\d+)\}",samp="sample"')


def parse_trees(root):
    """Parse all *_beast2.trees files under `root`.

    Returns list of (tree_id, total_tips, {location: tip_count}).
    """
    root = Path(root)
    stats = []
    for tf in sorted(root.rglob('*_beast2.trees')):
        batch  = tf.parent.name
        prefix = tf.stem.replace('_beast2', '')
        tid    = f'{batch}/{prefix}' if batch != root.name else prefix
        for idx, tree_str in enumerate(TREE_RE.findall(tf.read_text())):
            lc = defaultdict(int)
            for loc in TIP_RE.findall(tree_str):
                lc[int(loc)] += 1
            stats.append((f'{tid}_{idx}', sum(lc.values()), dict(lc)))
    return stats


def print_stats(stats):
    """Print min/max/mean for tree_width and subtree_width."""
    sizes = np.array([s[1] for s in stats])
    min_t = min(stats, key=lambda s: s[1])
    max_t = max(stats, key=lambda s: s[1])

    subs      = [(tid, loc, n) for tid, _, lc in stats for loc, n in lc.items()]
    sub_sizes = np.array([s[2] for s in subs])
    min_s     = min(subs, key=lambda s: s[2])
    max_s     = max(subs, key=lambda s: s[2])

    print(f'  tree_width     — Range: [{min_t[1]}, {max_t[1]}],  Mean: {sizes.mean():.1f}')
    print(f'    Min: {min_t[0]} ({min_t[1]} tips)')
    print(f'    Max: {max_t[0]} ({max_t[1]} tips)')
    print(f'  subtree_width  — Range: [{min_s[2]}, {max_s[2]}],  Mean: {sub_sizes.mean():.1f}')
    print(f'    Min: tree={min_s[0]}, loc={min_s[1]} ({min_s[2]} tips)')
    print(f'    Max: tree={max_s[0]}, loc={max_s[1]} ({max_s[2]} tips)')


# --- Original summary ----------------------------------------------------------
stats     = parse_trees(DATASET)
locations = sorted({loc for _, _, lc in stats for loc in lc})

print(f'Dataset:   {DATASET}')
print(f'Locations: {locations}')
print(f'\n=== All trees ({len(stats)} trees) ===')
print_stats(stats)

Dataset:   /Users/lukelyu/Desktop/epidata/3k/
Locations: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

=== All trees (3000 trees) ===
  tree_width     — Range: [253, 8528],  Mean: 2279.8
    Min: batch_0/261_0 (253 tips)
    Max: batch_0/378_0 (8528 tips)
  subtree_width  — Range: [11, 1391],  Mean: 228.0
    Min: tree=batch_0/261_0, loc=2 (11 tips)
    Max: tree=batch_2/959_0, loc=1 (1391 tips)


In [ ]:
# --- Filter by subtree_width (outbreak removal) --------------------------------
SUB_TOP_PCT = 1

# Percentile cutoff across all per-location tip counts
all_sub_sizes = np.array([n for _, _, lc in stats for n in lc.values()])
sub_cutoff = int(np.percentile(all_sub_sizes, 100 - SUB_TOP_PCT, method='lower'))

# Drop any tree with at least one location subtree exceeding the cutoff
filtered = [s for s in stats if all(n <= sub_cutoff for n in s[2].values())]

print(f'=== Discarding trees with any subtree_width > {sub_cutoff} '
      f'(top {SUB_TOP_PCT}% subtree threshold) ===')
print(f'    {len(stats)} -> {len(filtered)} trees remain\n')
print_stats(filtered)

max_subtree = max(n for _, _, lc in filtered for n in lc.values())
print(f'\n--num_locations {len(locations)} --subtree_width {max_subtree}')